# Random Semantic Algebra — Latent Search Demo
Search demo with **non-catalog semantic predicates**. CLIP image semantics define latent concepts such as `minimalist`, `office-appropriate`, `technical/sporty`, `retro`, `elegant`, `relaxed`, `chunky`, and `quiet luxury`. RSA compiles them into sparse LUT programs over a fixed **192-byte/item** code.

**Important:** the latent labels are CLIP-teacher-defined, not human annotations. This is a mechanism demo of latent predicate compilation/composition, not a production-quality fashion taxonomy.

In [ ]:
!pip -q install datasets transformers sentence-transformers scikit-learn pandas numpy pyarrow pillow ipywidgets matplotlib accelerate


In [ ]:
import torch
from transformers import CLIPModel

# Robust compatibility shim across Transformers CLIP API variants.
# Some releases return tensors; others return ModelOutput objects;
# and some already expose the projected 512-D feature in pooler_output.
if not hasattr(CLIPModel, '_rsa_orig_get_image_features'):
    CLIPModel._rsa_orig_get_image_features = CLIPModel.get_image_features
if not hasattr(CLIPModel, '_rsa_orig_get_text_features'):
    CLIPModel._rsa_orig_get_text_features = CLIPModel.get_text_features

def _rsa_project_feature_output(out, projection):
    if torch.is_tensor(out):
        return out

    # Prefer explicitly projected embeddings when available.
    for attr in ('image_embeds', 'text_embeds', 'pooler_output'):
        value = getattr(out, attr, None)
        if torch.is_tensor(value):
            d = value.shape[-1]
            if d == projection.out_features:
                return value
            if d == projection.in_features:
                return projection(value)

    if isinstance(out, (tuple, list)):
        for value in out:
            if torch.is_tensor(value) and value.ndim == 2:
                d = value.shape[-1]
                if d == projection.out_features:
                    return value
                if d == projection.in_features:
                    return projection(value)

    raise TypeError(f'Unsupported CLIP feature output type/shape: {type(out)}')

def _rsa_get_image_features(self, *args, **kwargs):
    out = self._rsa_orig_get_image_features(*args, **kwargs)
    return _rsa_project_feature_output(out, self.visual_projection)

def _rsa_get_text_features(self, *args, **kwargs):
    out = self._rsa_orig_get_text_features(*args, **kwargs)
    return _rsa_project_feature_output(out, self.text_projection)

# Always overwrite the wrappers so this cell also repairs an older bad shim
# in an already-running Colab session.
CLIPModel.get_image_features = _rsa_get_image_features
CLIPModel.get_text_features = _rsa_get_text_features
CLIPModel._rsa_feature_compat_patched = True
print('CLIP feature compatibility shim v2 installed')


In [ ]:
import base64, gzip, urllib.request
url='https://raw.githubusercontent.com/hanialshater/LSH_Memory/refs/heads/rsa-v2-colab/experiments/rsa_latent_search_demo.part0'
blob=urllib.request.urlopen(url).read().decode()
exec(gzip.decompress(base64.b64decode(blob)).decode())
